# RoadWatch AI: cleaning the pothole dataset
In this notebook, We go through the checks and split decisions used to prepare the pothole dataset, starting from `data/1` and verifying the final files in `data/cleaned`.

We use Pillow, ImageHash, NumPy and Matplotlib for the checks and plots. The paths are relative to the repository, so We can run the notebook from the project root or `notebooks/`.

We keep the original images and labels unchanged. Copying is disabled by default because the cleaned dataset already exists. We use the saved manifests to replay the recorded split decisions; the visual review itself is not automated.

The final v3 dataset contains **3,940 images, 10,111 boxes and 834 filename source groups**. The split is 3,196 images / 8,141 boxes for training, 375 / 1,097 for validation and 369 / 873 for testing. Below, We check these counts along with label validity and cross-split similarity.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, hashlib, math, random, shutil
import numpy as np
from PIL import Image, ImageDraw
import imagehash
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/1/train/images').is_dir()), None)
assert ROOT is not None, 'Open this notebook inside the repository containing data/1.'
RAW = ROOT / 'data/1'
SPLITS = ('train', 'valid', 'test')
WRITE_COPY = False
OUTPUT = ROOT / 'data/cleaned_notebook'  # I keep this separate from data/cleaned
print('Repository:', ROOT.name)


## 1. Exploring the data and pairing images with labels
We start by counting the images and labels in each original split. We match each image to a text file with the same filename stem, then check for missing labels, orphan labels and ambiguous names.

We also extract the part before `.rf.` as the filename source group. This lets me keep exported variants of the same named source together. Different source names can still refer to similar scenes, so We check that separately later.


In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
records = []
pairing_errors = []
for split in SPLITS:
    images = sorted(p for p in (RAW / split / 'images').iterdir()
                    if p.suffix.lower() in EXTENSIONS)
    labels = {p.stem: p for p in (RAW / split / 'labels').glob('*.txt')}
    stems = Counter(p.stem for p in images)
    pairing_errors += [(split, 'ambiguous image stem', s) for s,n in stems.items() if n > 1]
    pairing_errors += [(split, 'orphan label', s) for s in labels if s not in stems]
    for p in images:
        label = labels.get(p.stem)
        if label is None:
            pairing_errors.append((split, 'missing label', p.name))
        records.append(dict(image=p.name, original_split=split,
                            source=p.stem.split('.rf.')[0], path=p, label=label))
    print(split, len(images), 'images,', len(labels), 'labels')
assert not pairing_errors, pairing_errors[:20]
assert len({r['image'] for r in records}) == len(records), 'Ambiguous manifest image names'
by_name = {r['image']: r for r in records}
print('Images:', len(records), '| source groups:', len({r['source'] for r in records}))


## 2. Checking image integrity and YOLO annotations
We open each image to check that it can be decoded, then validate every nonempty label line against the YOLO format: `class x_center y_center width height`.

For this dataset, the class must be `0` for pothole. We check that all values are finite, the coordinates are normalized, the width and height are positive, and the box fits inside the image with a small allowance for rounding. We count empty labels separately. If a check fails, We stop and inspect the file instead of deleting it automatically.


In [ ]:
def read_boxes(path):
    boxes = []
    for line_number, line in enumerate(path.read_text().splitlines(), 1):
        if not line.strip():
            continue
        values = line.split()
        assert len(values) == 5, f'{path.name}:{line_number}: expected 5 fields'
        cls, x, y, w, h = map(float, values)
        assert all(math.isfinite(v) for v in (cls,x,y,w,h)), 'Nonfinite value'
        assert cls == 0, 'Expected pothole class 0'
        assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
        eps = 1e-6
        assert x-w/2 >= -eps and y-h/2 >= -eps
        assert x+w/2 <= 1+eps and y+h/2 <= 1+eps
        boxes.append((x,y,w,h))
    return boxes

errors = []
for r in records:
    try:
        with Image.open(r['path']) as im:
            im.verify()
        with Image.open(r['path']) as im:
            im.load()
            r['size'] = im.size
        r['boxes'] = read_boxes(r['label'])
    except Exception as exc:
        errors.append((r['image'], str(exc)))
print('Invalid images/labels:', len(errors))
assert not errors, errors[:20]
print('Boxes:', sum(len(r['boxes']) for r in records))
print('Empty labels:', sum(not r['boxes'] for r in records))


In [ ]:
# We overlay a few labels to inspect the boxes against the images.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, random.Random(42).sample(records, 3)):
    with Image.open(r['path']) as im:
        im = im.convert('RGB')
    draw = ImageDraw.Draw(im)
    W,H = im.size
    for x,y,w,h in r['boxes']:
        draw.rectangle(((x-w/2)*W, (y-h/2)*H, (x+w/2)*W, (y+h/2)*H), outline='lime', width=3)
    ax.imshow(im); ax.set_title(r['source']); ax.axis('off')
plt.show()


## 3. Checking exact duplicates and leakage in the original splits
We calculate a SHA-256 hash for each image to find files with identical bytes. We also check whether filename source groups or exact hashes appear in more than one original split.

These checks help me identify overlap without changing the original files. The final dataset retains all 3,940 images; the later split checks confirm that related filename groups stay together.


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
exact = defaultdict(list)
for r in records:
    r['sha256'] = sha256(r['path'])
    exact[r['sha256']].append(r)
duplicate_sets = [v for v in exact.values() if len(v) > 1]
print('Exact duplicate sets:', len(duplicate_sets))
print('Extra copies:', sum(len(v)-1 for v in duplicate_sets))
def leaking_groups(rows, key, split_key):
    groups = defaultdict(set)
    for r in rows:
        groups[r[key]].add(r[split_key])
    return {k: v for k,v in groups.items() if len(v) > 1}
print('Original source groups spanning splits:', len(leaking_groups(records, 'source', 'original_split')))
print('Original exact hashes spanning splits:', len(leaking_groups(records, 'sha256', 'original_split')))


## 4. Grouping the data for an 80/10/10 split
Here We shuffle the source groups with a fixed seed and allocate roughly 80% to training, 10% to validation and 10% to testing. We assign whole groups, so variants of one source cannot end up in different splits. Since groups contain different numbers of images, the image percentages can differ from the group percentages.

This cell produces a reproducible split proposal. To replay the completed cleaning process, We load the saved v2 manifest next. We do not have enough detail in the available logs to recreate the earlier transition from `split_manifest.csv` to v2, so We keep that distinction explicit.


In [ ]:
groups = sorted({r['source'] for r in records})
random.Random(42).shuffle(groups)
a, b = int(.8*len(groups)), int(.9*len(groups))
proposal = {g: ('train' if i < a else 'valid' if i < b else 'test')
            for i,g in enumerate(groups)}
print('Proposal images:', Counter(proposal[r['source']] for r in records))

def load_manifest(name):
    with (ROOT / 'data' / name).open(newline='') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == len(records)
    assert len({r['image'] for r in rows}) == len(rows)
    assert {r['image'] for r in rows} == set(by_name)
    for r in rows:
        raw = by_name[r['image']]
        assert r['source'] == raw['source']
        assert r['original_split'] == raw['original_split']
        assert int(r['annotations']) == len(raw['boxes'])
        assert r['new_split'] in SPLITS
    assert not leaking_groups(rows, 'source', 'new_split')
    return rows
v2 = load_manifest('split_manifest_v2.csv')
v3 = load_manifest('split_manifest_v3.csv')
print('Recorded v2 images:', Counter(r['new_split'] for r in v2))


## 5. Checking perceptual similarity and reviewing flagged pairs
We calculate a 64-bit pHash for each image using `hash_size=8`. We flag cross-split pairs with a Hamming distance of 5 or less, then display them side by side for review.

The hash helps me find visually similar images, including some that have different file bytes or source names. A flagged pair still needs visual review, and zero matches does not guarantee that every shared scene has been found.

For this run, We replay the recorded review decisions below and compare the result with v3. If We change the data or split proposal, We need to review the new flagged pairs before carrying those decisions forward.


In [ ]:
for r in records:
    with Image.open(r['path']) as im:
        r['phash'] = int(str(imagehash.phash(im, hash_size=8)), 16)

def cross_split_pairs(assignments):
    # About 7.8 million cheap integer comparisons for this dataset.
    pairs = []
    for i, left in enumerate(records):
        for right in records[i+1:]:
            if assignments[left['image']] == assignments[right['image']]:
                continue
            distance = (left['phash'] ^ right['phash']).bit_count()
            if distance <= 5:
                pairs.append((left['image'], right['image'], distance))
    return pairs
v2_assignment = {r['image']: r['new_split'] for r in v2}
flagged = cross_split_pairs(v2_assignment)
print('v2 cross-split pHash pairs:', len(flagged))
for pair in flagged:
    print(pair)


In [ ]:
# We display the flagged pairs for visual review before replaying the recorded decisions.
for left, right, distance in flagged:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, name in zip(axes, (left, right)):
        r = by_name[name]
        with Image.open(r['path']) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f"{r['source']} ({v2_assignment[name]})"); ax.axis('off')
    fig.suptitle(f'pHash distance {distance}')
    plt.show()


### Moving five test source groups to training
The v2 audit flagged eight cross-split pairs. The recorded review decision was to move **18_jpg, 355_jpg, potholes504_png, potholes618_png and potholes641_png** from test to train, affecting 16 images in total.

We apply that decision to whole source groups, then compare the resulting assignments with the v3 manifest. Validation stays at 375 images, while test decreases from 385 to 369. This cell replays the saved decision; it does not repeat the visual judgment automatically.


In [ ]:
MOVED_GROUPS = {'18_jpg', '355_jpg', 'potholes504_png', 'potholes618_png', 'potholes641_png'}
assert len(flagged) == 8, 'Review changed inputs before replaying historical decisions'
final = []
for row in v2:
    row = dict(row)
    if row['source'] in MOVED_GROUPS:
        assert row['new_split'] == 'test'
        row['new_split'] = 'train'
    final.append(row)
assert sum(r['source'] in MOVED_GROUPS for r in final) == 16
assert {r['image']: r for r in final} == {r['image']: r for r in v3}
assignment = {r['image']: r['new_split'] for r in final}
assert not leaking_groups(final, 'source', 'new_split')
assert all(len({assignment[r['image']] for r in group}) == 1 for group in duplicate_sets)
final_pairs = cross_split_pairs(assignment)
assert not final_pairs, final_pairs[:10]
assert len(records) == 3940 and len(groups) == 834
assert sum(len(r['boxes']) for r in records) == 10111
EXPECTED = {'train': (3196, 8141), 'valid': (375, 1097), 'test': (369, 873)}
for split in SPLITS:
    subset = [r for r in final if r['new_split'] == split]
    actual = (len(subset), sum(int(r['annotations']) for r in subset))
    assert actual == EXPECTED[split], (split, actual)
    print(split, actual[0], 'images;', actual[1], 'boxes')
print('Source leakage: 0; cross-split pHash matches <=5: 0')


## 6. Creating a separate cleaned copy
We leave `WRITE_COPY = False` for the normal verification run. When We enable it, the copy step writes to `data/cleaned_notebook`, leaving `data/1`, the saved manifests and the existing `data/cleaned` unchanged.

Before copying, We check any existing destination files against the originals. Matching files can be reused on a rerun; a conflict stops the copy. We also write a YOLO YAML without an absolute dataset path, keeping the copy portable.

From the repository root, We can use the separate copy with `model.train(data="data/cleaned_notebook/data.yaml")`.


In [ ]:
def copy_if_missing(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert sha256(source) == sha256(target), f'Existing output differs: {target}'
    else:
        shutil.copy2(source, target)

if WRITE_COPY:
    assert OUTPUT.resolve() == (ROOT / 'data/cleaned_notebook').resolve()
    assert not OUTPUT.is_symlink()
    # We check all destinations before starting the copy.
    jobs = []
    for r in final:
        raw = by_name[r['image']]
        for kind, source in [('images', raw['path']), ('labels', raw['label'])]:
            target = OUTPUT / r['new_split'] / kind / source.name
            assert OUTPUT.resolve() in target.resolve().parents, 'Unsafe output path'
            if target.exists():
                assert sha256(source) == sha256(target), f'Conflict: {target}'
            jobs.append((source, target))
    yaml_text = 'train: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames:\n  0: pothole\n'
    yaml_file = OUTPUT / 'data.yaml'
    if yaml_file.exists():
        assert yaml_file.read_text() == yaml_text, 'Existing YAML differs'
    for source, target in jobs:
        copy_if_missing(source, target)
    if not yaml_file.exists():
        yaml_file.write_text(yaml_text)
    print('Copied/verified', len(final), 'image-label pairs to', OUTPUT.name)
else:
    print('Read-only run: optional copy skipped.')


## 7. Validating the cleaned files
Finally, We check that each split contains exactly the expected images and labels. We compare the copied files byte-for-byte with the originals, decode the images again and revalidate the annotations.

Because the image bytes match, the hash and leakage results above also apply to the cleaned files. We check the existing `data/cleaned` when it is present and the separate notebook copy if it has been created.


In [ ]:
def validate_copy(folder):
    for split in SPLITS:
        rows = [r for r in final if r['new_split'] == split]
        expected_images = {r['image'] for r in rows}
        expected_labels = {by_name[r['image']]['label'].name for r in rows}
        assert {p.name for p in (folder/split/'images').iterdir() if p.is_file()} == expected_images
        assert {p.name for p in (folder/split/'labels').iterdir() if p.is_file()} == expected_labels
        count = 0
        for r in rows:
            raw = by_name[r['image']]
            image = folder/split/'images'/r['image']
            label = folder/split/'labels'/raw['label'].name
            assert sha256(image) == raw['sha256']
            assert sha256(label) == sha256(raw['label'])
            with Image.open(image) as im:
                im.load()
            count += len(read_boxes(label))
        assert (len(rows), count) == EXPECTED[split]
        print(folder.name, split, len(rows), 'images,', count, 'boxes: passed')

for folder in [ROOT/'data/cleaned', OUTPUT]:
    if folder.exists():
        validate_copy(folder)
    else:
        print(folder.name, 'not present; skipped')
print('FINAL DATASET VALIDATION PASSED')


## Final checks
We verify the expected v3 counts, replay the five-group correction and check that filename groups stay within one split. The final pHash audit checks for cross-split matches at distance 5 or less, while the file comparison confirms that the cleaned data matches the originals.

Filename grouping and pHash are still heuristics. We use them alongside visual review, rather than treating them as proof of scene independence or annotation quality. We keep the saved manifests as the record of the split decisions and reserve the test split for final evaluation.
